# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [1952, 1957, 1962, 1967, 1972, 1977, 1982, 1987, 1992, 1997, 2002, 2007]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    425.9
std      17.3     12.9  1.061579e+08     9857.5    248.3
min    1952.0     23.6  6.001100e+04      241.2      4.0


## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2000 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [3]:
# Task 1 — Europe: GDP vs Life Expectancy, 2000 vs 2007
df_europe = df[df['continent'] == 'Europe'].copy()
df_task1 = df_europe[df_europe['year'].isin([2000, 2007])].copy()
df_task1['year'] = df_task1['year'].astype(str)

fig1 = px.scatter(
    df_task1,
    x='gdpPercap',
    y='lifeExp',
    color='year',
    color_discrete_map={'2000': '#4393c3', '2007': '#d6604d'},
    size='pop',
    size_max=30,
    log_x=True,
    hover_name='country',
    title='Europe got richer and lived longer: 2000 → 2007',
    labels={
        'gdpPercap': 'GDP per Capita (log scale, USD)',
        'lifeExp': 'Life Expectancy (years)',
        'year': 'Year',
        'pop': 'Population',
    },
)
fig1.update_traces(marker=dict(opacity=0.85, line=dict(width=0.5, color='white')))
fig1.update_layout(height=500, legend_title_text='Year')
fig1.show()

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [4]:
# Task 2 — Bubble chart: Asia highlighted, all others grey (2007)
import numpy as np

df_2007 = df[df['year'] == 2007].copy()
HIGHLIGHT = 'Asia'
df_2007['group'] = df_2007['continent'].apply(lambda c: c if c == HIGHLIGHT else 'Other')

# Render "Other" first so Asia bubbles sit on top
df_2007 = df_2007.sort_values('group', key=lambda s: s.map({'Other': 0, HIGHLIGHT: 1}))

fig2 = px.scatter(
    df_2007,
    x='gdpPercap',
    y='lifeExp',
    size='pop',
    size_max=60,
    color='group',
    color_discrete_map={HIGHLIGHT: '#e6550d', 'Other': '#cccccc'},
    log_x=True,
    hover_name='country',
    hover_data={'pop': ':,.0f', 'gdpPercap': ':,.0f', 'group': False},
    title="Asia's stark divide: oil-rich cities and agrarian giants share one continent (2007)",
    labels={
        'gdpPercap': 'GDP per Capita (log scale, USD)',
        'lifeExp': 'Life Expectancy (years)',
        'pop': 'Population',
        'group': '',
    },
)
fig2.update_traces(selector=dict(name='Other'), marker=dict(opacity=0.4, line=dict(width=0)))
fig2.update_traces(selector=dict(name=HIGHLIGHT), marker=dict(opacity=0.75, line=dict(width=0.5, color='white')))

# --- annotations (manually positioned to avoid overlap) ---
china = df_2007[df_2007['country'] == 'China'].iloc[0]
fig2.add_annotation(
    x=np.log10(china['gdpPercap']),
    y=china['lifeExp'],
    xref='x', yref='y',
    text='<b>China</b><br>1.3 bn people,<br>mid-tier GDP',
    showarrow=True, arrowhead=2, arrowcolor='#e6550d',
    ax=70, ay=-55,
    font=dict(size=10, color='#333333'),
    bgcolor='white', bordercolor='#e6550d', borderwidth=1, borderpad=4,
    xanchor='left',
)

japan = df_2007[df_2007['country'] == 'Japan'].iloc[0]
fig2.add_annotation(
    x=np.log10(japan['gdpPercap']),
    y=japan['lifeExp'],
    xref='x', yref='y',
    text='<b>Japan</b><br>Highest life exp.',
    showarrow=True, arrowhead=2, arrowcolor='#e6550d',
    ax=-80, ay=-30,
    font=dict(size=10, color='#333333'),
    bgcolor='white', bordercolor='#e6550d', borderwidth=1, borderpad=4,
    xanchor='right',
)

fig2.update_layout(
    height=580,
    legend_title_text='',
    legend=dict(itemsizing='constant'),
)
fig2.show()